# 02 - Embeddings (Grant Witness)
data source: grant-witness.us

### Imports

In [1]:
# imports
import pandas as pd
import re
import os
from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report

import torch
from transformers import AutoTokenizer
from adapters import AutoAdapterModel


In [2]:
df_epa = pd.read_csv('00_data/df_epa.csv')
df_nih = pd.read_csv('00_data/df_nih.csv')
df_nsf = pd.read_csv('00_data/df_nsf.csv')
df_samhsa = pd.read_csv('00_data/df_samhsa.csv')
df_cdc = pd.read_csv('00_data/df_cdc.csv')

# Embeddings

In [3]:
def pick_device():
    if torch.backends.mps.is_available():
        return 'mps'
    if torch.cuda.is_available():
        return 'cuda'
    return 'cpu'

device = pick_device()
print(device)

mps


In [4]:
dfs_abstract = {'nih': df_nih, 'nsf': df_nsf, 'epa': df_epa}
dfs_titles = {'nih': df_nih, 'nsf': df_nsf, 'epa': df_epa, 'samhsa': df_samhsa, 'cdc': df_cdc}

### all-MiniLM-L6-v2
https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2

In [7]:
df_nsf.columns

Index(['Unnamed: 0', 'grant_id', 'status', 'terminated', 'suspended',
       'termination_date', 'termination_indicator', 'reinstated',
       'reinstatement_date', 'reinstatement_indicator', 'cruz_list', 'nsf_url',
       'usaspending_url', 'project_title', 'abstract', 'org_name', 'org_state',
       'org_city', 'award_type', 'usasp_start_date', 'usasp_end_date',
       'start_date_original', 'end_date_original', 'nsf_program_name',
       'nsf_primary_program', 'usasp_nsf_office', 'award_amount',
       'nsf_obligated', 'usasp_total_obligated', 'usasp_obligation_hist',
       'usasp_total_obligated_corrected', 'usasp_outlaid', 'estimated_budget',
       'award_outlaid', 'award_remaining', 'post_termination_deobligation',
       'division', 'directorate', 'div', 'dir', 'record_sha1', 'agency',
       'flagged_words_pen', 'flagged_words_nyt', 'flagged_words_all',
       'has_flagged_word', 'num_flagged_words', 'num_pen_words',
       'num_nyt_words'],
      dtype='object')

In [13]:
allminilm_model = SentenceTransformer('all-MiniLM-L6-v2', device=device)

def add_column_embeddings(df, name, model, text_col='abstract', id_col='grant_id', batch_size=32):
    print(f'Extracting {text_col} embeddings for {name}...')

    texts = df[text_col].fillna('').astype(str).tolist()

    vectors = model.encode(texts, batch_size=batch_size, show_progress_bar=True, normalize_embeddings=True)

    vectors = np.asarray(vectors)

    np.savez(f'embeddings/{name}_{text_col}_allminilm.npz', grant_id=df[id_col].values, embeddings=vectors)

    print(f'Saved: embeddings/{name}_{text_col}_allminilm.npz')

In [15]:
for name, df in dfs_titles.items():
    add_column_embeddings(df, name, allminilm_model, text_col='abstract')

Extracting abstract embeddings for nih...


Batches:   0%|          | 0/183 [00:00<?, ?it/s]

Saved: embeddings/nih_abstract_allminilm.npz
Extracting abstract embeddings for nsf...


Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Saved: embeddings/nsf_abstract_allminilm.npz
Extracting abstract embeddings for epa...


Batches:   0%|          | 0/21 [00:00<?, ?it/s]

Saved: embeddings/epa_abstract_allminilm.npz
Extracting abstract embeddings for samhsa...


Batches:   0%|          | 0/92 [00:00<?, ?it/s]

Saved: embeddings/samhsa_abstract_allminilm.npz
Extracting abstract embeddings for cdc...


KeyError: 'abstract'

### nomic-embed-text-v1.5
https://huggingface.co/nomic-ai/nomic-embed-text-v1.5



todo: run on gpu

In [ ]:
nomic_model = SentenceTransformer('nomic-ai/nomic-embed-text-v1.5', trust_remote_code=True, device=device)

def add_nomic_embeddings(df, text_col='abstract', embedding_col='embeddings_nomic', batch_size=8, task_prefix='classification'):
    texts = df[text_col].fillna('').astype(str).tolist()
    texts = [f"{task_prefix}: {t}" for t in texts]

    vectors = nomic_model.encode(texts, batch_size=batch_size, show_progress_bar=True, normalize_embeddings=True)

    df[embedding_col] = [v.tolist() for v in vectors]
    return df

for name, df in dfs.items():
    add_nomic_embeddings(df)
    df.to_parquet(f'../data/embeddings/{name}_with_embeddings_nomic.parquet', index=False)

### specter2_aug2023refresh
https://huggingface.co/allenai/specter2_aug2023refresh

- specific for scientific tasks
- title + abstract

todo: run on gpu



In [ ]:
specter_tokenizer = AutoTokenizer.from_pretrained('allenai/specter2_aug2023refresh_base')
specter_model = AutoAdapterModel.from_pretrained('allenai/specter2_aug2023refresh_base')
specter_model.load_adapter('allenai/specter2_aug2023refresh_classification', source='hf', load_as='specter2_classification', set_active=True)

specter_model.to(device)
specter_model.eval()

@torch.no_grad()
def specter2_classification_embeddings(titles, abstracts, batch_size=16, max_length=512):
    all_embeddings = []

    for start in range(0, len(titles), batch_size):
        batch_titles = titles[start : start + batch_size]
        batch_abstracts = abstracts[start : start + batch_size]

        text_batch = [(t or '') + specter_tokenizer.sep_token + (a or '') for t, a in zip(batch_titles, batch_abstracts)]
        inputs = specter_tokenizer(text_batch, padding=True, truncation=True, max_length=max_length, return_tensors='pt', return_token_type_ids=False)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        outputs = specter_model(**inputs)

        batch_embeddings = outputs.last_hidden_state[:, 0, :]
        all_embeddings.append(batch_embeddings.detach().cpu().numpy())

    embeddings = np.vstack(all_embeddings)

    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    embeddings = embeddings / np.clip(norms, 1e-12, None)

    return embeddings


def add_specter2_classification_embeddings(df, title_column='project_title', abstract_column='abstract', embedding_column='abstract_embedding_specter2_cls', batch_size=16):
    titles = df[title_column].fillna('').astype(str).tolist()
    abstracts = df[abstract_column].fillna('').astype(str).tolist()

    vectors = specter2_classification_embeddings(titles=titles, abstracts=abstracts, batch_size=batch_size)

    df[embedding_column] = [vec.tolist() for vec in vectors]
    return df


for name, df in dfs.items():
    add_specter2_classification_embeddings(df)
    df.to_parquet(f'../data/embeddings/{name}_with_embeddings_specter2.parquet', index=False)